# LLM Training Pipeline — Exercise

> **Learning objectives:** Understand the three-stage LLM training pipeline: pretraining → instruction fine-tuning (SFT) → preference alignment (DPO/RLHF).

This notebook demonstrates **LoRA fine-tuning**, **SFT vs DPO comparison**, and **perplexity tracking** across training stages.

**Tools:**
- `transformers` — load pretrained models
- `peft` — parameter-efficient fine-tuning (LoRA)
- `trl` — SFT and DPO trainers
- `datasets` — small training sets
- All local, GPU optional (CPU works but slower)

In [ ]:
## 0 · Setup
import subprocess, sys
pkgs = ["transformers", "peft", "trl", "datasets", "torch", "accelerate", "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)
print("✓ Packages installed")

# Check GPU availability
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device: {device}")
if device == "cpu":
    print("  ⚠️  Training on CPU will be slow. Consider using a smaller model or fewer steps.")

## Exercise 1 · LoRA Fine-Tuning

**Task:** Load GPT-2, apply LoRA adapters, fine-tune on 100 instruction examples, then compare base vs fine-tuned outputs.

**LoRA (Low-Rank Adaptation)** freezes the pretrained weights and injects small trainable matrices into attention layers:

$$
W' = W + \alpha \cdot BA
$$

where:
- $W$ = frozen pretrained weights
- $B, A$ = trainable low-rank matrices ($r \ll d$)
- $\alpha$ = scaling factor

This reduces trainable parameters from ~124M (GPT-2) to ~0.3M (~0.2%).

### Steps:
1. Load GPT-2 base model
2. Prepare a small instruction dataset (100 examples)
3. Apply LoRA configuration
4. Fine-tune for 50 steps
5. Compare outputs before vs after fine-tuning

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# 1. Load GPT-2 base model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
if device == "cpu":
    model = model.to(device)

print(f"✓ Loaded {model_name}")
print(f"  Total parameters: {model.num_parameters():,}")

# TODO: Test base model output on a sample prompt
prompt = "Explain photosynthesis in simple terms:"
# TODO: Generate output from base model and print it
# Hint: Use tokenizer.encode() and model.generate()

In [ ]:
# 4. Fine-tune for 50 steps
from transformers import TrainingArguments, Trainer

# TODO: Prepare dataset for causal LM training
# Hint: Tokenize text and create labels

# TODO: Set up TrainingArguments with:
#   - output_dir="./lora-gpt2"
#   - num_train_epochs=1
#   - max_steps=50
#   - per_device_train_batch_size=4
#   - learning_rate=2e-4
#   - logging_steps=10

# TODO: Create Trainer and train
# Hint: Use Trainer with model, args, and train_dataset

In [ ]:
# 5. Compare outputs
# TODO: Generate output from fine-tuned model on the same prompt
# TODO: Print side-by-side comparison:
#   - Base model output
#   - Fine-tuned model output
# TODO: Observe how instruction-following behavior has changed

## Exercise 2 · SFT vs DPO Comparison

**Task:** Compare Supervised Fine-Tuning (SFT) and Direct Preference Optimization (DPO).

**SFT (Supervised Fine-Tuning):** Train on (instruction, response) pairs using standard cross-entropy loss.

**DPO (Direct Preference Optimization):** Train on preference pairs (preferred vs rejected) without a reward model:

$$
\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x,y_w,y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]
$$

where:
- $y_w$ = preferred (winning) response
- $y_l$ = rejected (losing) response
- $\pi_\theta$ = policy being trained
- $\pi_{\text{ref}}$ = reference model (frozen)
- $\beta$ = temperature controlling strength of KL constraint

### Steps:
1. Load a base model
2. Fine-tune with SFT on instruction-response pairs
3. Fine-tune with DPO on preference pairs
4. Compare outputs on ambiguous prompts

In [ ]:
from trl import SFTTrainer, DPOTrainer

# 1. Load base model (reuse from Exercise 1 or load fresh)
# TODO: Load model and tokenizer

# 2. SFT fine-tuning
# TODO: Load a small instruction dataset (e.g., alpaca[:50])
# TODO: Set up SFTTrainer with TrainingArguments
# TODO: Train for 30 steps
# TODO: Save model as "./sft-model"

In [ ]:
# 3. DPO fine-tuning
# TODO: Load a preference dataset
# Hint: Use load_dataset("Anthropic/hh-rlhf", split="train[:50]")
# Format: {"prompt": "...", "chosen": "...", "rejected": "..."}

# TODO: Load a fresh base model as reference
# TODO: Set up DPOTrainer with:
#   - model (trainable)
#   - ref_model (frozen reference)
#   - train_dataset (preference pairs)
#   - beta=0.1
# TODO: Train for 30 steps
# TODO: Save model as "./dpo-model"

In [ ]:
# 4. Compare outputs
test_prompts = [
    "Should I prioritize speed or accuracy in this ML model?",
    "Write a short story about a robot learning to paint.",
    "Explain the trolley problem."
]

# TODO: Load all three models (base, SFT, DPO)
# TODO: Generate outputs for each prompt from all three models
# TODO: Print side-by-side comparison
# TODO: Observe how SFT improves instruction-following,
#       and DPO affects preference alignment (helpfulness, safety)

## Exercise 3 · Perplexity Tracking

**Task:** Track perplexity across training stages to understand how pretraining → SFT → RLHF affects model confidence.

**Perplexity** measures how well a model predicts a sequence:

$$
\text{PPL}(X) = \exp \left( -\frac{1}{N} \sum_{i=1}^N \log p(x_i | x_{<i}) \right)
$$

- Lower perplexity = better prediction = higher confidence
- Typically: pretraining decreases PPL, SFT slightly increases it (domain shift), RLHF increases it more (optimizing for preference, not likelihood)

### Steps:
1. Load checkpoints from different training stages (simulate if needed)
2. Compute perplexity on a held-out test set
3. Plot PPL curve across training stages
4. Interpret how each stage affects model confidence

In [ ]:
import torch
from torch.nn import CrossEntropyLoss
import matplotlib.pyplot as plt

def compute_perplexity(model, tokenizer, texts):
    """Compute perplexity on a list of text samples."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            # TODO: Tokenize text and create input_ids and labels
            # Hint: labels = input_ids.clone(), shift happens in model

            # TODO: Get model outputs (logits)

            # TODO: Compute cross-entropy loss
            # Hint: Use CrossEntropyLoss() on flattened logits and labels

            # TODO: Accumulate loss and token count
            pass

    # TODO: Calculate average loss and convert to perplexity
    # perplexity = exp(avg_loss)
    return 0.0  # Replace with actual perplexity

# Load test set
# TODO: Load a small test dataset (e.g., wikitext-2-raw-v1, test split[:100])

In [ ]:
# Simulate or load checkpoints from different stages
# For this exercise, we'll use:
#   - Base GPT-2 (pretrained)
#   - SFT checkpoint (from Exercise 2)
#   - DPO checkpoint (from Exercise 2)

# TODO: Load each checkpoint
# TODO: Compute perplexity on test set for each
# TODO: Store results in a dict: {"stage": perplexity_value}

In [ ]:
# Plot perplexity curve
# TODO: Create a bar plot or line plot showing:
#   - x-axis: training stage (Base, SFT, DPO)
#   - y-axis: perplexity
# TODO: Add title and labels
# TODO: Display plot

# Expected pattern:
#   - Base: lowest PPL (optimized for next-token prediction)
#   - SFT: slightly higher (domain shift to instructions)
#   - DPO: higher still (optimized for preference, not likelihood)

## Summary

**What you learned:**
1. **LoRA** reduces trainable parameters by ~99% while maintaining fine-tuning quality
2. **SFT** improves instruction-following; **DPO** aligns with human preferences without a reward model
3. **Perplexity** increases across training stages as models optimize for different objectives (pretraining → instruction → preference)

**Key takeaways:**
- Pretraining: maximize likelihood on web text
- SFT: maximize likelihood on instruction-response pairs
- RLHF/DPO: maximize preference-based reward (trade-off with likelihood)

Each stage serves a different goal, and perplexity is not the only metric that matters—alignment and safety matter too.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# 1. Load GPT-2 base model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
if device == "cpu":
    model = model.to(device)

print(f"✓ Loaded {model_name}")
print(f"  Total parameters: {model.num_parameters():,}")

# TODO: Test base model output on a sample prompt
prompt = "Explain photosynthesis in simple terms:"
# TODO: Generate output from base model and print it
# Hint: Use tokenizer.encode() and model.generate()